In [3]:
using BenchmarkTools

In [1]:
struct Spectrum
    x::Float64
    y::Float64
    z::Float64
end

struct SurfaceInteraction
    x::Float64
    y::Float64
end

function SurfaceInteraction()
    return SurfaceInteraction(0,0)
end

const TextureType = Union{Float64, Spectrum}
abstract type AbstractTexture end
struct ConstantTexture{T <: TextureType} <: AbstractTexture
    value::T
end
function (c::ConstantTexture{T})(si::SurfaceInteraction)::T where T <: TextureType
    return c.value
end

function Base.zero(::Type{Spectrum})
    return Spectrum(0, 0, 0)
end

abstract type Image end
struct MIPMap{T <: TextureType} <: Image
    val::Matrix{T}
    function MIPMap(x::T) where T <: TextureType
        return new{T}(fill(x, 10, 10))
    end
end

struct ImageMap{T <: TextureType} <: Image
    val::Matrix{T}
    function ImageMap(x::T) where T <: TextureType
        return new{T}(fill(x, 10, 10))
    end
end

struct ImageTexture{T <: TextureType} <: AbstractTexture
    mipmap::Union{ImageMap{T}, MIPMap{T}}
    function ImageTexture(x::T, flag::Int8) where T <: TextureType
        if flag == Int8(0)
            return new{T}(ImageMap(x))
        else
            return new{T}(MIPMap(x))
        end
    end
end


abstract type AbstractMaterial end
struct MatteMaterial <: AbstractMaterial
    Kd::AbstractTexture  # really this should be spectral
    sigma::AbstractTexture  # really this should be float
    function MatteMaterial(Kd::AbstractTexture, sigma::AbstractTexture)
        si = SurfaceInteraction()
        @assert typeof(Kd(si)) == Spectrum
        @assert typeof(sigma(si)) == Float64
        return new(Kd, sigma)
    end
end

In [4]:
i = ImageTexture(1.0, Int8(0))
@btime ImageTexture(1.0, Int8(0))

  58.687 ns (2 allocations: 912 bytes)


ImageTexture{Float64}(ImageMap{Float64}([1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0; … ; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0]))

In [5]:
x,y = size(i.mipmap.val)
n = x * y * 8

800

In [6]:
i = ImageTexture(Spectrum(0.5, 0.5, 0.5), Int8(0))
@btime ImageTexture(Spectrum(0.5, 0.5, 0.5), Int8(0))

  190.724 ns (2 allocations: 2.52 KiB)


ImageTexture{Spectrum}(ImageMap{Spectrum}(Spectrum[Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5) … Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5); Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5) … Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5); … ; Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5) … Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5); Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5) … Spectrum(0.5, 0.5, 0.5) Spectrum(0.5, 0.5, 0.5)]))

In [7]:
x, y = size(i.mipmap.val)
n = x * y * 3 * 8

2400

In [8]:
m1 = MatteMaterial(ConstantTexture(Spectrum(0.5, 0.5, 0.5)), ConstantTexture(1.0))
si = SurfaceInteraction(10.0, 20.0)

SurfaceInteraction(10.0, 20.0)

In [9]:
@code_warntype MatteMaterial(ConstantTexture(Spectrum(0.5, 0.5, 0.5)), ConstantTexture(1.0))

MethodInstance for MatteMaterial(::ConstantTexture{Spectrum}, ::ConstantTexture{Float64})
  from MatteMaterial(Kd::AbstractTexture, sigma::AbstractTexture) @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:60
Arguments
  #ctor-self#::Core.Const(MatteMaterial)
  Kd::ConstantTexture{Spectrum}
  sigma::ConstantTexture{Float64}
Locals
  si::SurfaceInteraction
  @_5::ConstantTexture{Float64}
  @_6::ConstantTexture{Spectrum}
Body::MatteMaterial
1 ──       (si = Main.SurfaceInteraction())
│    %2  = (Kd)(si::Core.Const(SurfaceInteraction(0.0, 0.0)))::Spectrum
│    %3  = Main.typeof(%2)::Core.Const(Spectrum)
│    %4  = (%3 == Main.Spectrum)::Core.Const(true)
└───       goto #3 if not %4
2 ──       goto #4
3 ──       Core.Const(:(Base.AssertionError("typeof(Kd(si)) == Spectrum")))
└───       Core.Const(:(Base.throw(%7)))
4 ┄─ %9  = (sigma)(si::Core.Const(SurfaceInteraction(0.0, 0.0)))::Float64
│    %10 = Main.typeof(%9)::Core.Const(Float64)
│    %11 = (%10 == Main.

In [10]:
@code_warntype m1.Kd(si)

MethodInstance for (::ConstantTexture{Spectrum})(::SurfaceInteraction)
  from (c::ConstantTexture{T})(si::SurfaceInteraction) where T<:TextureType @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:21
Static Parameters
  T = Spectrum
Arguments
  c::ConstantTexture{Spectrum}
  si::SurfaceInteraction
Locals
  @_3::Spectrum
Body::Spectrum
1 ─ %1 = $(Expr(:static_parameter, 1))::Core.Const(Spectrum)
│   %2 = Base.getproperty(c, :value)::Spectrum
│        (@_3 = %2)
│   %4 = (@_3 isa %1)::Core.Const(true)
└──      goto #3 if not %4
2 ─      goto #4
3 ─      Core.Const(:(Base.convert(%1, @_3)))
└──      Core.Const(:(@_3 = Core.typeassert(%7, %1)))
4 ┄      return @_3



In [11]:
@btime m1.Kd(si)

  35.549 ns (1 allocation: 32 bytes)


Spectrum(0.5, 0.5, 0.5)

In [12]:
m = MatteMaterial(ConstantTexture(Spectrum(0.5, 0.5, 0.5)), ConstantTexture(1.0))
@btime MatteMaterial(ConstantTexture(Spectrum(0.5, 0.5, 0.5)), ConstantTexture(1.0))

  1.500 ns (0 allocations: 0 bytes)


MatteMaterial(ConstantTexture{Spectrum}(Spectrum(0.5, 0.5, 0.5)), ConstantTexture{Float64}(1.0))

In [14]:
struct Spectrum
    x::Float64
    y::Float64
    z::Float64
end

abstract type AbstractMaterial end
abstract type Texture end
abstract type FloatTexture <: Texture end
abstract type SpectrumTexture <: Texture end
struct ConstantFloatTexture <: FloatTexture
    value::Float64
end
struct ConstantSpectrumTexture <: SpectrumTexture
    value::Spectrum
end
struct MatteMaterial2{S <: SpectrumTexture, F <: FloatTexture} <: AbstractMaterial
    Kd::S
    sigma::F
end

function (c::ConstantFloatTexture)(si::SurfaceInteraction)
    return c.value
end
function (c::ConstantSpectrumTexture)(si::SurfaceInteraction)
    return c.value
end


In [15]:
m2 = MatteMaterial2(ConstantSpectrumTexture(Spectrum(0.5, 0.5, 0.5)), ConstantFloatTexture(1.0))

MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture}(ConstantSpectrumTexture(Spectrum(0.5, 0.5, 0.5)), ConstantFloatTexture(1.0))

In [16]:
@code_warntype MatteMaterial2(ConstantSpectrumTexture(Spectrum(0.5, 0.5, 0.5)), ConstantFloatTexture(1.0))

MethodInstance for MatteMaterial2(::ConstantSpectrumTexture, ::ConstantFloatTexture)
  from MatteMaterial2(Kd::S, sigma::F) where {S<:SpectrumTexture, F<:FloatTexture} @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:18
Static Parameters
  S = ConstantSpectrumTexture
  F = ConstantFloatTexture
Arguments
  #self#::Type{MatteMaterial2}
  Kd::ConstantSpectrumTexture
  sigma::ConstantFloatTexture
Body::MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture}
1 ─ %1 = Core.apply_type(Main.MatteMaterial2, $(Expr(:static_parameter, 1)), $(Expr(:static_parameter, 2)))::Core.Const(MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture})
│   %2 = %new(%1, Kd, sigma)::MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture}
└──      return %2



In [17]:
@btime m2.Kd(si)

  67.791 ns (3 allocations: 112 bytes)


Spectrum(0.5, 0.5, 0.5)

In [18]:
@code_warntype m2.Kd(si)

MethodInstance for (::ConstantSpectrumTexture)(::SurfaceInteraction)
  from (c::ConstantSpectrumTexture)(si::SurfaceInteraction) @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:25
Arguments
  c::ConstantSpectrumTexture
  si::SurfaceInteraction
Body::Spectrum
1 ─ %1 = Base.getproperty(c, :value)::Spectrum
└──      return %1



In [19]:
@btime m2.sigma(si)

  70.552 ns (3 allocations: 80 bytes)


1.0